# AliPOS table-booking discovery

This notebook is read-only. It uses official AliPOS credentials only for authentication, and its safety checks block the deployed restaurant ID from all probes.

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any, Mapping, Optional, Tuple
from urllib.parse import urlsplit


LIVE_FLAG_NAME = "ALLOW_LIVE_ALIPOS_READS"
UUID_PATTERN = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[1-5][0-9a-fA-F]{3}-[89abAB][0-9a-fA-F]{3}-[0-9a-fA-F]{12}"
UUID_RE = re.compile(rf"^{UUID_PATTERN}$")


class ProbeSafetyError(RuntimeError):
    pass


class LiveProbesDisabled(ProbeSafetyError):
    pass


def find_repo_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".env").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise ProbeSafetyError("Repository root with .env and notebooks was not found")


def read_dotenv(path: Path) -> dict:
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        values[key] = value
    return values


def _stream_output_text(cell: Mapping[str, Any]) -> str:
    parts = []
    for output in cell.get("outputs", []):
        text = output.get("text", "")
        if isinstance(text, list):
            parts.extend(str(item) for item in text)
        elif text:
            parts.append(str(text))
    return "".join(parts)


def extract_dummy_identifiers(source_notebook: Path) -> Tuple[str, Tuple[str, ...]]:
    document = json.loads(source_notebook.read_text(encoding="utf-8"))
    cells = document.get("cells", [])
    if len(cells) < 10:
        raise ProbeSafetyError("Source notebook does not contain cells 4 and 10")

    config_output = _stream_output_text(cells[3])
    restaurant_match = re.search(rf"Restaurant ID:\s*({UUID_PATTERN})", config_output)
    if restaurant_match is None:
        raise ProbeSafetyError("Dummy restaurant ID was not found in source cell 4")

    created_output = _stream_output_text(cells[9])
    marker = "ID созданных заказов / Yaratilgan buyurtma ID lari:"
    if marker not in created_output:
        raise ProbeSafetyError("Dummy order marker was not found in source cell 10")
    tail = created_output.split(marker, 1)[1].lstrip()
    try:
        order_map, _ = json.JSONDecoder().raw_decode(tail)
    except (TypeError, ValueError) as exc:
        raise ProbeSafetyError("Dummy order map in source cell 10 is invalid") from exc

    order_ids = tuple(order_map.get(name, "") for name in ("cash", "online-order"))
    if any(not UUID_RE.fullmatch(value) for value in order_ids):
        raise ProbeSafetyError("Source cell 10 does not contain two valid dummy order IDs")
    return restaurant_match.group(1), order_ids


def build_probe_config(
    repo_root: Path,
    environ: Optional[Mapping[str, str]] = None,
) -> dict:
    dotenv = read_dotenv(repo_root / ".env")
    process = dict(os.environ if environ is None else environ)

    def configured(name: str) -> str:
        return process.get(name) or dotenv.get(name, "")

    dummy_restaurant_id, dummy_order_ids = extract_dummy_identifiers(
        repo_root / "notebooks" / "alipos_support_report_ru_uz.ipynb"
    )
    return {
        "base_url": configured("ALIPOS_API_BASE_URL").rstrip("/"),
        "client_id": configured("ALIPOS_API_CLIENT_ID"),
        "client_secret": configured("ALIPOS_API_CLIENT_SECRET"),
        "deployed_restaurant_id": configured("ALIPOS_RESTAURANT_ID"),
        "dummy_restaurant_id": dummy_restaurant_id,
        "dummy_order_ids": dummy_order_ids,
        "live_enabled": process.get(LIVE_FLAG_NAME) == "1",
        "timeout_seconds": 15,
        "minimum_interval_seconds": 0.25,
        "max_requests": 120,
    }


def validate_probe_config(
    config: Mapping[str, Any],
    require_live: bool = True,
) -> None:
    required = (
        "base_url",
        "client_id",
        "client_secret",
        "deployed_restaurant_id",
        "dummy_restaurant_id",
    )
    missing = [name for name in required if not str(config.get(name, "")).strip()]
    if missing:
        raise ProbeSafetyError("Required AliPOS configuration is missing")

    parsed = urlsplit(str(config["base_url"]))
    hostname = (parsed.hostname or "").lower()
    if parsed.scheme != "https" or not hostname:
        raise ProbeSafetyError("AliPOS base URL must use HTTPS")
    if hostname != "alipos.uz" and not hostname.endswith(".alipos.uz"):
        raise ProbeSafetyError("Configured host is not an AliPOS host")
    if parsed.username or parsed.password:
        raise ProbeSafetyError("AliPOS base URL must not contain credentials")

    deployed_id = str(config["deployed_restaurant_id"])
    dummy_id = str(config["dummy_restaurant_id"])
    if not UUID_RE.fullmatch(deployed_id) or not UUID_RE.fullmatch(dummy_id):
        raise ProbeSafetyError("Restaurant IDs must be UUIDs")
    if deployed_id.casefold() == dummy_id.casefold():
        raise ProbeSafetyError("Dummy restaurant ID matches the deployed restaurant ID")

    order_ids = tuple(config.get("dummy_order_ids", ()))
    if len(order_ids) != 2 or any(not UUID_RE.fullmatch(str(value)) for value in order_ids):
        raise ProbeSafetyError("Exactly two valid dummy order IDs are required")
    if require_live and not bool(config.get("live_enabled")):
        raise LiveProbesDisabled(f"Set {LIVE_FLAG_NAME}=1 only for the approved live run")


In [ ]:
import hashlib
from typing import Sequence


SENSITIVE_KEY_TOKENS = {
    "address",
    "authorization",
    "card",
    "coordinate",
    "coordinates",
    "credential",
    "credentials",
    "cvv",
    "email",
    "lat",
    "latitude",
    "lng",
    "longitude",
    "number",
    "otp",
    "pan",
    "payment",
    "phone",
    "secret",
    "token",
}
SENSITIVE_KEY_COMPOUNDS = {
    "cardnumber",
    "clientid",
    "clientname",
    "clientsecret",
    "customername",
    "paymentinfo",
}
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")
UZ_PHONE_RE = re.compile(r"\+?998[0-9 ()-]{9,16}")
BEARER_RE = re.compile(r"(?i)\bBearer\s+[A-Za-z0-9._~-]+")
UUID_TEXT_RE = re.compile(UUID_PATTERN)


def fingerprint_identifier(value: str) -> str:
    digest = hashlib.sha256(value.encode("utf-8")).hexdigest()[:10]
    return f"id:{digest}"


def sanitize_text(text: str) -> str:
    value = UUID_TEXT_RE.sub(lambda match: fingerprint_identifier(match.group(0)), str(text))
    value = BEARER_RE.sub("Bearer [REDACTED]", value)
    value = EMAIL_RE.sub("[REDACTED_EMAIL]", value)
    value = UZ_PHONE_RE.sub("[REDACTED_PHONE]", value)
    return value[:500]


def _is_sensitive_key(key: str) -> bool:
    separated = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", str(key))
    parts = [part.casefold() for part in re.findall(r"[A-Za-z0-9]+", separated)]
    compact = "".join(parts)
    return any(part in SENSITIVE_KEY_TOKENS for part in parts) or compact in SENSITIVE_KEY_COMPOUNDS


def redact_value(value: Any, key: str = "") -> Any:
    if key and _is_sensitive_key(key):
        return "[REDACTED]"
    if isinstance(value, Mapping):
        return {
            str(item_key): redact_value(item_value, str(item_key))
            for item_key, item_value in value.items()
        }
    if isinstance(value, (list, tuple)):
        return [redact_value(item) for item in value]
    if isinstance(value, str):
        if UUID_RE.fullmatch(value):
            return fingerprint_identifier(value)
        return sanitize_text(value)
    return value


def _case_insensitive_get(mapping: Mapping[str, Any], name: str, default: Any) -> Any:
    wanted = name.casefold()
    for key, value in mapping.items():
        if str(key).casefold() == wanted:
            return value
    return default


def _safe_collection_metadata(items: Sequence[Any]) -> dict:
    item_fields = []
    if items and isinstance(items[0], Mapping):
        item_fields = sorted(str(key) for key in items[0].keys())
    return {"count": len(items), "item_fields": item_fields}


def _safe_hall_table_preview(payload: Mapping[str, Any]) -> dict:
    halls = _case_insensitive_get(payload, "halls", [])
    tables = _case_insensitive_get(payload, "tables", [])
    safe_halls = []
    safe_tables = []
    for hall in halls if isinstance(halls, list) else []:
        if not isinstance(hall, Mapping):
            continue
        safe_halls.append(
            {
                "id": redact_value(_case_insensitive_get(hall, "id", "")),
                "title": sanitize_text(str(_case_insensitive_get(hall, "title", ""))),
                "servicePercent": _case_insensitive_get(hall, "servicePercent", None),
            }
        )
    for table in tables if isinstance(tables, list) else []:
        if not isinstance(table, Mapping):
            continue
        safe_tables.append(
            {
                "id": redact_value(_case_insensitive_get(table, "id", "")),
                "title": sanitize_text(str(_case_insensitive_get(table, "title", ""))),
                "hallId": redact_value(_case_insensitive_get(table, "hallId", "")),
            }
        )
    return {"halls": safe_halls, "tables": safe_tables}


def summarize_payload(payload: Any, route_name: str = "") -> dict:
    if isinstance(payload, Mapping):
        collections = {
            str(key): _safe_collection_metadata(value)
            for key, value in payload.items()
            if isinstance(value, list)
        }
        summary = {
            "type": "object",
            "fields": sorted(str(key) for key in payload.keys()),
            "collections": collections,
        }
        if route_name == "halls_and_tables":
            summary["preview"] = _safe_hall_table_preview(payload)
        return summary
    if isinstance(payload, list):
        return {
            "type": "array",
            "count": len(payload),
            "item_fields": (
                sorted(str(key) for key in payload[0].keys())
                if payload and isinstance(payload[0], Mapping)
                else []
            ),
        }
    if payload is None:
        return {"type": "none"}
    return {"type": type(payload).__name__}


def extract_hall_table_ids(payload: Any) -> Tuple[Optional[str], Optional[str]]:
    if not isinstance(payload, Mapping):
        return None, None
    halls = _case_insensitive_get(payload, "halls", [])
    tables = _case_insensitive_get(payload, "tables", [])

    def first_id(items: Any) -> Optional[str]:
        if not isinstance(items, list):
            return None
        for item in items:
            if not isinstance(item, Mapping):
                continue
            value = _case_insensitive_get(item, "id", "")
            if isinstance(value, str) and UUID_RE.fullmatch(value):
                return value
        return None

    return first_id(halls), first_id(tables)


def classify_result(result: Mapping[str, Any]) -> str:
    status = result.get("status")
    content_type = str(result.get("content_type", "")).casefold()
    allow = {part.strip().upper() for part in str(result.get("allow", "")).split(",") if part.strip()}
    if status in (401, 403):
        return "unauthorized/forbidden"
    if status in (400, 409, 422):
        return "invalid test data"
    if status == 404:
        return "unsupported"
    if status == 405:
        return "ambiguous" if {"GET", "OPTIONS"} & allow else "unsupported"
    if isinstance(status, int) and 200 <= status < 300:
        if "json" in content_type and result.get("payload") is not None:
            return "confirmed"
        return "ambiguous"
    return "ambiguous"


def summarize_result(result: Mapping[str, Any]) -> dict:
    error = result.get("error")
    return {
        "name": str(result.get("name", "")),
        "family": str(result.get("family", "")),
        "method": str(result.get("method", "")),
        "path": sanitize_text(str(result.get("path", ""))),
        "status": result.get("status"),
        "classification": classify_result(result),
        "latency_ms": result.get("latency_ms"),
        "content_type": sanitize_text(str(result.get("content_type", ""))),
        "allow": sanitize_text(str(result.get("allow", ""))),
        "shape": summarize_payload(result.get("payload"), str(result.get("name", ""))),
        "error": "[REDACTED]" if error else "",
    }


In [ ]:
MAX_TOTAL_REQUESTS = 120
OPTIONS_RESERVE = 12
BOOKING_TERMS = (
    "tableBooking",
    "table-booking",
    "tableReservation",
    "table-reservation",
    "booking",
    "bookings",
    "reservation",
    "reservations",
    "availability",
    "tableAvailability",
    "table-availability",
    "bookingAvailability",
    "booking-availability",
    "table",
    "tables",
    "hall",
    "halls",
    "floor",
    "floors",
    "reserve",
    "reserves",
    "bron",
)
ID_SCOPED_TERMS = BOOKING_TERMS


def _route(name: str, family: str, path: str, method: str = "GET") -> dict:
    return {"name": name, "family": family, "method": method, "path": path}


def _deduplicate_routes(routes: Sequence[Mapping[str, str]]) -> list:
    selected = []
    seen = set()
    for route in routes:
        key = (str(route["method"]), str(route["path"]))
        if key in seen:
            continue
        seen.add(key)
        selected.append(dict(route))
    return selected


def build_documented_routes(restaurant_id: str) -> list:
    return [
        _route("restaurants", "documented", "/restaurants"),
        _route("payment_methods", "documented", "/api/Integration/v1/paymentMethod/all"),
        _route(
            "menu_composition",
            "documented",
            f"/api/Integration/v1/menu/{restaurant_id}/composition",
        ),
        _route(
            "halls_and_tables",
            "documented",
            f"/api/Integration/v1/restaurant/{restaurant_id}/halls-and-tables",
        ),
    ]


def build_order_routes(order_ids: Sequence[str]) -> list:
    routes = []
    for index, order_id in enumerate(order_ids, start=1):
        routes.append(
            _route(
                f"dummy_order_{index}",
                "order_read",
                f"/api/Integration/v1/order/{order_id}",
            )
        )
        routes.append(
            _route(
                f"dummy_order_status_{index}",
                "order_read",
                f"/api/Integration/v1/order/{order_id}/status",
            )
        )
    return routes


def build_booking_routes(
    restaurant_id: str,
    hall_id: Optional[str] = None,
    table_id: Optional[str] = None,
) -> list:
    routes = []
    for term in BOOKING_TERMS:
        routes.extend(
            [
                _route(f"{term}_top", "top_level", f"/api/Integration/v1/{term}"),
                _route(
                    f"{term}_restaurant_scoped",
                    "restaurant_scoped",
                    f"/api/Integration/v1/restaurant/{restaurant_id}/{term}",
                ),
                _route(
                    f"{term}_restaurant_argument",
                    "argument",
                    f"/api/Integration/v1/{term}/{restaurant_id}",
                ),
                _route(
                    f"{term}_menu_scoped",
                    "menu_scoped",
                    f"/api/Integration/v1/menu/{restaurant_id}/{term}",
                ),
            ]
        )

    for term in ID_SCOPED_TERMS:
        if table_id:
            routes.extend(
                [
                    _route(
                        f"{term}_table_argument",
                        "id_scoped",
                        f"/api/Integration/v1/{term}/{table_id}",
                    ),
                    _route(
                        f"{term}_restaurant_table",
                        "id_scoped",
                        f"/api/Integration/v1/restaurant/{restaurant_id}/{term}/{table_id}",
                    ),
                ]
            )
        if hall_id:
            routes.extend(
                [
                    _route(
                        f"{term}_hall_argument",
                        "id_scoped",
                        f"/api/Integration/v1/{term}/{hall_id}",
                    ),
                    _route(
                        f"{term}_restaurant_hall",
                        "id_scoped",
                        f"/api/Integration/v1/restaurant/{restaurant_id}/{term}/{hall_id}",
                    ),
                ]
            )
    return _deduplicate_routes(routes)


def select_get_routes(
    candidates: Sequence[Mapping[str, str]],
    completed_count: int,
) -> list:
    available = max(0, MAX_TOTAL_REQUESTS - OPTIONS_RESERVE - completed_count)
    return [dict(route) for route in candidates[:available]]


def build_option_routes(
    get_results: Sequence[Mapping[str, Any]],
    completed_count: int,
) -> list:
    available = max(0, MAX_TOTAL_REQUESTS - completed_count)
    method_failures = [result for result in get_results if result.get("status") == 405]
    representatives = []
    represented_families = set()
    for result in get_results:
        family = str(result.get("family", ""))
        if result.get("status") != 404 or family in represented_families:
            continue
        represented_families.add(family)
        representatives.append(result)

    routes = [
        _route(
            f"options_{result.get('name', 'route')}",
            str(result.get("family", "")),
            str(result.get("path", "")),
            method="OPTIONS",
        )
        for result in (*method_failures, *representatives)
    ]
    return _deduplicate_routes(routes)[:available]
